# OpenJev

Jevの発表から数日のうちに、「OpenJev」などを名乗るJevのオープンソース再現・代替実装がGitHubやHugging Faceに大量に登場した。

Jev本体のアーキテクチャや学習方法は非公開のため、どれも **Jevのインターフェース（state + Choice/Score/Noul → 型付きの確率）を真似たもの** であり、中身は実装ごとにまったく違う。同じ「OpenJev」という名前でも別物である点に注意。

:::{card} 参考
- [Jev alternatives: open-source reproductions, local models and classifiers | System One Models](https://systemonemodels.org/examples/alternatives/)
- [OpenJevを読んで、非生成AIでの代替案を考えてみた | DevelopersIO](https://dev.classmethod.jp/en/articles/openjev-non-generative-ai-alternatives/)
- awesome系リスト：[AbdelStark/awesome-typesafe-jev](https://github.com/AbdelStark/awesome-typesafe-jev)、[cobanov/awesome-jev](https://github.com/cobanov/awesome-jev)、[v-modal/awesome-jev-tools](https://github.com/v-modal/awesome-jev-tools)
:::

## アプローチの分類

実装はおおむね次のように分けられる。

| アプローチ | 仕組み | 学習 | 確率の性質 |
|---|---|---|---|
| **凍結LLMのlogitを読む** | 選択肢のラベル（A/B/Cなど）のトークン位置のlogitにsoftmaxをかける。1回のforward（prefill）だけで生成はしない | 不要 | キャリブレーションされていない（温度スケーリングで補正するものもある） |
| **凍結LLM + 判断ヘッド** | 凍結したLLMの特徴量の上に小さなヘッドやLoRAを学習する | 軽い | 学習次第 |
| **エンコーダーのfine-tune** | ModernBERTなどの双方向エンコーダーに型付きの判断ヘッドを付けて学習 | 必要 | 温度フィッティングでECEを報告しているものもある |
| **スクラッチ学習** | 独自の小型アーキテクチャを一から学習 | 必要 | 学習次第 |
| **拡散モデル** | DiffusionGemmaなどの最終デノイズステップの分布を読む | 不要 | － |
| **ゼロショット分類器** | NLIベースのゼロショット分類器などで代用 | 不要 | 分布の集中度のみ |

多くの実装は TypeSafe と同じ `POST /v1/systemone` のワイヤーフォーマットを実装しており、TypeSafe公式SDKの接続先を変えるだけで使える。

## 「OpenJev」を名乗るリポジトリ

| リポジトリ | ベースモデル | 概要 |
|---|---|---|
| [razorback16/openjev](https://github.com/razorback16/openjev) | DiffusionGemma 26B-A4B | Jev互換のAPIを持つ判断サーバー。読み取りだけの1パスでラベルトークンの確率分布を得る。vLLM（NVIDIA 24GB以上）またはMLX（Apple Silicon）で動作。選択肢255個まで・画像入力にも対応 |
| [GPT-AGI/OpenJev](https://github.com/GPT-AGI/OpenJev) | Qwen2.5 など | 回答位置のlogitに、ユーザーが与えたラベルだけでsoftmaxをかける（1回のforwardでJSONのパースも不要）。Claude Code風のREPLでconfidenceとレイテンシを表示。キャリブレーション・判断ヘッドの学習はロードマップ上の予定 |
| [TheoLeeCJ/SemIf (openjev)](https://github.com/TheoLeeCJ/openjev) | Qwen3.5-4B（凍結） | 「semantic if」をRTX 3090クラスで自宅で動かす。凍結モデルのlogitを読む。旧名OpenJev |
| [daseinlabs/open-jev](https://github.com/daseinlabs/open-jev) | Gemma 3 4B | 文脈を1回prefillしてKVキャッシュを選択肢ぶん複製し、全候補を1回のバッチforwardでスコアリング。凍結したGemmaの特徴量の上にタスクごとのヘッドをMLXで学習できる（listwise交差エントロピー）。Apple Siliconで約90ms |
| [zhangcy122/OpenJev](https://github.com/zhangcy122/OpenJev) | vLLM/SGLangのオープンモデル、Laya、Jev本体 | 複数のバックエンドを束ねるゲートウェイ（OpenJevPro）。曖昧なクエリは推論モデル（System 2）に回し、その結果を高速なルール（System 1）に蒸留する「Explore First, Crystallize Later」の仕組み。選択肢を個別にスコアリングして順序不変性を保証 |
| [ekzhang/openjev-sglang](https://github.com/ekzhang/openjev-sglang) | Qwen3.6-35B-A3B | SGLang上のprefillだけで動くJev互換エンドポイント。logitベースで、キャリブレーションされていないと明記。B200クラスのGPU / Modalにデプロイ |
| [kyegomez/open-jev](https://github.com/kyegomez/open-jev) | 独自（PyTorch） | Jevのアイデアを第一原理から再構成した研究実装。双方向Transformerエンコーダーでstateを1回エンコードし、Noul/Choice/Scoreの型付きreadoutヘッドで並列に回答。RLCDを模した損失（ソフトターゲットの尤度+キャリブレーション+一貫性）を持つが、重みはランダムで学習済みモデルはない |
| [ikermoel/open-alternative-jev](https://github.com/ikermoel/open-alternative-jev) | Qwen系（任意のオープンウェイトLLM） | 各質問をチャットターンとして1つの系列に連結し、プレースホルダー位置のlogitだけを読む（1回のforward）。HF Transformers / vLLM対応。温度スケーリングと選択肢の並べ替えによる位置バイアス緩和も実装。ベンチマークではゼロショットのQwen3.6-27Bが73.7%で、Jev API（72.7%）と同程度 |
| [SiliconLabAI/OpenJev](https://github.com/SiliconLabAI/OpenJev) | － | 「OpenSource Jev」とだけ説明がある |

## その他の再現・代替実装

| リポジトリ | ベースモデル | 概要 |
|---|---|---|
| [jaredpalmer/kev](https://github.com/jaredpalmer/kev) | Qwen3.5（0.8B / 4B / 9B）+ LoRA | 凍結したベースモデルの上に判断ヘッドを学習。学習コード・評価データも公開。CUDA / ROCm / MLX |
| [TianyuCodings/NanoJev](https://github.com/TianyuCodings/NanoJev) | 独自 0.6B | 並列の判断モデル。出力トークンを一切デコードせずに確率分布を返す。学習パイプライン・重み・データセットを公開 |
| [NandhaKishorM/laya](https://github.com/NandhaKishorM/laya) | ModernBERT-large（421M、英語）/ mmBERT-base（322M、多言語） | 非自己回帰の判断モデル。エンコーダーを全体fine-tune。温度フィッティング後のECE 0.081。T4で約35ms、CPUでも動く |
| [wfzyx/von](https://github.com/wfzyx/von) | ModernBERT（395M） | 各選択肢を独立にスコアリングして正規化。`/v1/systemone` を提供。GPUで約18ms、CPUでも可 |
| [Mapika/decider](https://github.com/Mapika/decider) | Qwen3.5-2B-Base | fine-tuneしたモデルで1回のforwardから確率分布を得る。Jevのワイヤーフォーマットを実装 |
| [zhengxuyu/litjev](https://github.com/zhengxuyu/litjev) | Qwen | 判断層で選択肢をスコアリングする独立な再現実装 |
| [nokia-applied-research/AnyJev](https://github.com/nokia-applied-research/AnyJev) | 任意のオープンLLM | fine-tuneなしでlogitや隠れ状態から型付きの答えを読み出す |
| [r-ms/mini-jev](https://github.com/r-ms/mini-jev) | Qwen3-4B-Instruct（凍結） | 選択肢の文字の次トークンlogitを読む。再現ではないと明記 |
| [heman10x-ngu/openjev-verdict-2.0](https://github.com/heman10x-ngu/openjev-verdict-2.0) | 独自 151M | 独自に学習した非自己回帰の判断モデル。ECE 0.0144を報告 |
| [vinnylarouge/jevlike](https://github.com/vinnylarouge/jevlike) | 独自（バイト埋め込み + option attention） | スクラッチから学習した小型モデル。CPU / Apple Silicon / CUDA |
| [taeold/djev-run](https://github.com/taeold/djev-run) | DiffusionGemma | 最終デノイズステップのlogit分布を読む。Cloud Runにデプロイできる |
| [hr98w/jev-visual](https://github.com/hr98w/jev-visual) | Qwen3.5-0.8B（MLX） | 画像を入力とする判断。logitスコアリング |
| [NullPo-jp/PocketJev](https://github.com/NullPo-jp/PocketJev) | 小型VLM | iPhone上で動くSwiftUIアプリ。A/B/Cの次トークンlogitを使う（キャリブレーションなし） |
| [deepanwadhwa/OpenDecision](https://github.com/deepanwadhwa/OpenDecision) | ModernBERT-large-zeroshot-v2.0 | NLIベースのゼロショット分類でChoice/Score/Noulを実現。学習不要、CPUで動く |

## 所感・まとめ

- 「凍結LLMのlogitを読む」実装が多い。LLMの **prefillだけで生成しない** ので速く、出力が選択肢に限られるので型エラーも起きない。Jevの「並列に答える」「型安全」という性質はこれでかなり再現できる
- 一方で、Jevの売りである **キャリブレーション（RLCD）** は再現が難しく、logitをそのまま使う実装の多くは「確率はモデルの好みであって、キャリブレーションされていない」と注意書きしている
- 温度スケーリングで補正したり、ECEを報告したりしている実装もある
- 選択肢の並び順で答えが変わる **位置バイアス** への対策（選択肢を個別にスコアリング、並べ替えて平均）も実装ごとの差が出るポイント
- ModernBERTなどのエンコーダーをfine-tuneする方向は、従来の「テキスト分類モデル」をSystem One風のAPIで包んだものと見ることもできる